Importing modules

In [66]:
import numpy as np

Defining parameters

In [67]:
np.random.seed(10)

n_users = 35        # secondary users
n_primary = 5       # primary channels
n_channels = 30     # available spectrum bands

noise = 0.1         # noise for SINR

PS_penalty = 100    # penalty for primary-econdary user channel collision
SS_penalty = 80     # penalty for secondary-secondary user channel collision

N=35                # particles
S=n_channels-1      # search space [0, n_channels-1]
D=n_users           # dimensions = number of users
n_iter = 500        # number of updates of positions

a=0.7
b=1.5
b_hat=1.5
c=0.1

beta_start = 1
beta_end = 0.5




Getting SINR from random

In [68]:
# Simulate channel gains randomly (in reality this comes from pathloss model)
# SINR[i][j] = signal quality if user i uses channel j

channel_gain = np.random.uniform(0.1, 1.0, (n_users, n_channels))     # to get SINR which gets throughput
SINR = channel_gain / noise                                           # simplified, no inter-user interference yet


Setting up Primary Users

In [69]:
# Primary user occupancy: PU[j] = 1 means channel j is occupied by primary user

PU_occupied = np.zeros(n_channels, dtype=int)
PU_occupied[np.random.choice(n_channels, n_primary, replace=False)] = 1

Fitness function which gets the throughput using SINR and adds penalty

In [70]:
def fitness(x,PS_penalty = PS_penalty, SS_penalty = SS_penalty):
    channels = np.clip(np.round(x).astype(int), 0, n_channels - 1)
    throughput = 0
    penalty = 0
    for user in range(n_users):
        ch = channels[user]
        if PU_occupied[ch]:
            penalty += PS_penalty                                 # hard penalty for interfering with primary user
        else:
            throughput += np.log2(1 + SINR[user, ch])
    
    for ch in range(n_channels):
        users_on_ch = np.sum(channels == ch)
        if users_on_ch > 1:
            penalty += (users_on_ch - 1) * SS_penalty             # secondary user collision penalty
    
    return -throughput + penalty




PSO algorithm

In [71]:
def dpso(f, D, N, S, n_iter, a, b, b_hat, c):

    x = np.random.uniform(0, S, size=(N, D))                 # setting up the initial positions of the N number of particles
    v = np.random.normal(size=(N, D))                        # setting up the initial velocities
    p = x.copy()                                             # best particle position
    fp = np.array([f(p[i]) for i in range(N)])               # throughput of all particles
    p_hat = x[np.argmin(fp)].copy()                          # global best position
    fp_hat = f(p_hat)                                        # throughput of global best position


    for i in range (n_iter):

        if i%(n_iter//10)== 0:                               # to show progress
            print(f"progress {(i/n_iter)*100:.0f}%")

        r,r_hat = np.random.uniform(0, 1, (2,N, D))          # setting up random parameters

        v = a*v + b*r*(p-x) + b_hat*r_hat*(p_hat-x)          # updating velocities as vector sum of the directions of initial velocity, local minima, local maxima
        x = x + c*v                                          # updating position according to velocities
        x = np.clip(x, 0, S)                                 # to limit the range within the subspace

        for n in range(N):                                   # calculation for each particle

            xn = x[n]                                        # getting position of that particle           
            fxn = f(xn)                                      # current throughput of that particle
            fpn = fp[n]                                      # best throughput of that particle

            if fxn < fpn:                                    # if current throughput of that particle is better the previous ones, update
                p[n] = xn.copy()
                fp[n] = fxn

                if fxn < fp_hat:                             # if the current througput is the global best throughput, update
                    p_hat = xn.copy()
                    fp_hat = fxn
    
    print("progress 100%")
    return p_hat                                             # "coordinates", ie channel allocation of global best throughput

        



Calling PSO and printing throughput

In [72]:
result = dpso(fitness, D, N, S, n_iter, a, b, b_hat, c)

C_best_assignment = np.clip(np.round(result).astype(int), 0, n_channels-1)

C_throughput = 0
for user in range(n_users):
    ch = C_best_assignment[user]
    if not PU_occupied[ch]:
        C_throughput += np.log2(1 + SINR[user, ch])

print("Channel assignment:", C_best_assignment)
print("Throughput:", C_throughput)

progress 0%
progress 10%
progress 20%
progress 30%
progress 40%
progress 50%
progress 60%
progress 70%
progress 80%
progress 90%
progress 100%
Channel assignment: [18  1 16 25  5  9 25  7 19 15  7 21 12 21 15 22 11 22 22 14  5  4 15  8
 21  3 10  6 17 15 25 26  4 18 27]
Throughput: 85.40230309121267


QPSO algorithm

In [73]:
def dqpso(f, D, N, S, n_iter, beta_start, beta_end):

    x = np.random.uniform(0, S, size=(N, D))                            # setting up the initial positions of the N number of particles
    p = x.copy()                                                        # best particle position
    fp = np.array([f(p[i]) for i in range(N)])                          # throughput of all particles
    p_hat = x[np.argmin(fp)].copy()                                     # global best position
    fp_hat = f(p_hat)                                                   # throughput of global best position


    for i in range(n_iter):

        if i%(n_iter//10)== 0:                                          # to show progress
            print(f"progress {(i/n_iter)*100:.0f}%")
        
        
        beta = beta_start - (beta_start - beta_end) * i / n_iter        # Beta decreases linearly from beta_start to beta_end

        mbest = np.mean(p, axis=0)                                      # Mean best position ie average of all personal bests

        phi = np.random.uniform(0,1, (N,D))                             
        p_local = phi * p + (1 - phi) * p_hat                           # local attractor for each particle (works like velocity or inertia)

        u = np.random.uniform(0, 1, (N,D))                           
        sign = 2 * np.random.randint(0, 2, size=(N,D)) - 1
        x = p_local + sign * beta * np.abs(mbest - x) * np.log(1/u)     # calculates the next value of x
        x = np.clip(x, 0, S)                                            # to limit the range within the subspace


        for n in range(N):                                              # calculation for each particle

            xn = x[n]                                                   # getting position of that particle           
            fxn = f(xn)                                                 # current throughput of that particle
            fpn = fp[n]                                                 # best throughput of that particle

            if fxn < fpn:                                               # if current throughput of that particle is better the previous ones, update
                p[n] = xn.copy()
                fp[n] = fxn

                if fxn < fp_hat:                                        # if the current througput is the global best throughput, update
                    p_hat = xn.copy()
                    fp_hat = fxn
    
    print("progress 100%")
    return p_hat                                                        # "coordinates", ie channel allocation of global best throughput

            

    

Calling QPSO and printing throughput

In [74]:
result_qpso = dqpso(fitness, D, N, S, n_iter, beta_start, beta_end)

Q_best_assignment = np.clip(np.round(result_qpso).astype(int), 0, n_channels-1)

Q_throughput = 0
for user in range(n_users):
    ch = Q_best_assignment[user]
    if not PU_occupied[ch]:
        Q_throughput += np.log2(1 + SINR[user, ch])

print("Channel assignment:", Q_best_assignment)
print("Throughput:", Q_throughput)

progress 0%
progress 10%
progress 20%
progress 30%
progress 40%
progress 50%
progress 60%
progress 70%
progress 80%
progress 90%
progress 100%
Channel assignment: [18 17 15 28  8 10  0 21  1 21  6 14  5 14  3 10  4  4  6 11 12 25 26 22
 29 28  4  9  2  8 18 16 27  0 11]
Throughput: 96.79975426847884
